ResNet

In [2]:
import torch

# all nn libraries nn.layer, convs and loss functions
import torch.nn as nn

# display Image
from IPython.display import Image

# visualisation
!pip install torchview
import torchvision
from torchview import draw_graph

# other torch functions needed
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.utils.data as data
import torch.optim as optim

import time
from tqdm import tqdm  # for progress bars
from google.colab import drive

drive.mount('/content/drive')

train_dir = "/content/drive/My Drive/datasets/train"
test_dir = "/content/drive/My Drive/datasets/test"

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # resize images to 224x224
    transforms.ToTensor(),  # convert images to tensors
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)

# create DataLoader for batching
train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
test_loader = data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

# print class-to-index mapping, ensure that data is loaded properly
print("Class-to-Index Mapping:", train_dataset.class_to_idx)

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

Mounted at /content/drive
Class-to-Index Mapping: {'Apple': 0, 'Apricot': 1, 'Ban Mian': 2, 'Bibimbap': 3, 'Blackberry': 4, 'Burger': 5, 'Chinese fritters': 6, 'Claypot Rice': 7, 'Duck Rice': 8, 'Fish Ball Noodles': 9, 'Fish and chips': 10, 'Hor Fun': 11, 'Indian Prata': 12, 'Kway Teow': 13, 'Laksa': 14, 'Lor mee': 15, 'Mee rebus': 16, 'Mee siam': 17, 'Miso ramen, with fishcake': 18, 'Nuts': 19, 'Papaya': 20, 'Porridge': 21, 'Prawn Noodle': 22, 'Seafood Noodles Soup': 23, 'Soft boiled eggs': 24, 'Udon': 25, 'ayam penyet': 26, 'bak kut teh': 27, 'bak kwa': 28, 'banana': 29, 'bee hoon': 30, 'black pepper crab': 31, 'blueberries': 32, 'cheese fries': 33, 'chicken rice': 34, 'chicken soup': 35, 'chilli crab': 36, 'chwee kueh': 37, 'cooked brown rice': 38, 'cooked white rice': 39, 'corn': 40, 'curry puff': 41, 'dumpling': 42, 'fish head curry': 43, 'fried chicken': 44, 'goreng pisang': 45, 'green leafy vegetables': 46, 'har cheong gai': 47, 'hokkien prawn mee': 48, 'ice kacang': 49, 'kebab 

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [3]:
model_parameters={}
model_parameters['resnet18'] = ([64,128,256,512],[2,2,2,2],1,False)
# model_parameters['resnet34'] = ([64,128,256,512],[3,4,6,3],1,False)
# model_parameters['resnet50'] = ([64,128,256,512],[3,4,6,3],4,True)
# model_parameters['resnet101'] = ([64,128,256,512],[3,4,23,3],4,True)
# model_parameters['resnet152'] = ([64,128,256,512],[3,8,36,3],4,True)

In [4]:
class Bottleneck(nn.Module):

    def __init__(self,in_channels,intermediate_channels,expansion,is_Bottleneck,stride):

        """
        Creates a Bottleneck with conv 1x1->3x3->1x1 layers.

        Note:
          1. Addition of feature maps occur at just before the final ReLU with the input feature maps
          2. if input size is different from output, select projected mapping or else identity mapping.
          3. if is_Bottleneck=False (3x3->3x3) are used else (1x1->3x3->1x1). Bottleneck is required for resnet-50/101/152
        Args:
            in_channels (int) : input channels to the Bottleneck
            intermediate_channels (int) : number of channels to 3x3 conv
            expansion (int) : factor by which the input #channels are increased
            stride (int) : stride applied in the 3x3 conv. 2 for first Bottleneck of the block and 1 for remaining

        Attributes:
            Layer consisting of conv->batchnorm->relu
        """

        super(Bottleneck,self).__init__()

        self.expansion = expansion
        self.in_channels = in_channels
        self.intermediate_channels = intermediate_channels
        self.is_Bottleneck = is_Bottleneck

        if self.in_channels==self.intermediate_channels*self.expansion:
            self.identity = True
        else:
            self.identity = False
            projection_layer = []
            projection_layer.append(nn.Conv2d(in_channels=self.in_channels, out_channels=self.intermediate_channels*self.expansion, kernel_size=1, stride=stride, padding=0, bias=False ))
            projection_layer.append(nn.BatchNorm2d(self.intermediate_channels*self.expansion))
            self.projection = nn.Sequential(*projection_layer)

        self.relu = nn.ReLU()

        if self.is_Bottleneck:
            # bottleneck
            # 1x1
            self.conv1_1x1 = nn.Conv2d(in_channels=self.in_channels, out_channels=self.intermediate_channels, kernel_size=1, stride=1, padding=0, bias=False )
            self.batchnorm1 = nn.BatchNorm2d(self.intermediate_channels)

            # 3x3
            self.conv2_3x3 = nn.Conv2d(in_channels=self.intermediate_channels, out_channels=self.intermediate_channels, kernel_size=3, stride=stride, padding=1, bias=False )
            self.batchnorm2 = nn.BatchNorm2d(self.intermediate_channels)

            # 1x1
            self.conv3_1x1 = nn.Conv2d(in_channels=self.intermediate_channels, out_channels=self.intermediate_channels*self.expansion, kernel_size=1, stride=1, padding=0, bias=False )
            self.batchnorm3 = nn.BatchNorm2d( self.intermediate_channels*self.expansion )

        else:
            # basicblock
            # 3x3
            self.conv1_3x3 = nn.Conv2d(in_channels=self.in_channels, out_channels=self.intermediate_channels, kernel_size=3, stride=stride, padding=1, bias=False )
            self.batchnorm1 = nn.BatchNorm2d(self.intermediate_channels)

            # 3x3
            self.conv2_3x3 = nn.Conv2d(in_channels=self.intermediate_channels, out_channels=self.intermediate_channels, kernel_size=3, stride=1, padding=1, bias=False )
            self.batchnorm2 = nn.BatchNorm2d(self.intermediate_channels)

    def forward(self,x):

        in_x = x

        if self.is_Bottleneck:
            x = self.relu(self.batchnorm1(self.conv1_1x1(x)))
            x = self.relu(self.batchnorm2(self.conv2_3x3(x)))
            x = self.batchnorm3(self.conv3_1x1(x))

        else:
            x = self.relu(self.batchnorm1(self.conv1_3x3(x)))
            x = self.batchnorm2(self.conv2_3x3(x))

        if self.identity:
            x += in_x
        else:
            x += self.projection(in_x)

        x = self.relu(x)

        return x

In [5]:
class ResNet(nn.Module):

    def __init__(self, resnet_variant,in_channels,num_classes):
        """
        Creates the ResNet architecture based on the provided variant. 18/34/50/101 etc.
        Based on the input parameters, define the channels list, repeatition list along with expansion factor(4) and stride(3/1)
        using _make_blocks method, create a sequence of multiple Bottlenecks
        Average Pool at the end before the FC layer

        Args:
            resnet_variant (list) : eg. [[64,128,256,512],[3,4,6,3],4,True]
            in_channels (int) : image channels (3)
            num_classes (int) : output #classes

        Attributes:
            Layer consisting of conv->batchnorm->relu

        """
        super(ResNet,self).__init__()
        self.channels_list = resnet_variant[0]
        self.repeatition_list = resnet_variant[1]
        self.expansion = resnet_variant[2]
        self.is_Bottleneck = resnet_variant[3]

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False )
        self.batchnorm1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()

        self.maxpool = nn.MaxPool2d(kernel_size=3,stride=2,padding=1)

        self.block1 = self._make_blocks( 64 , self.channels_list[0], self.repeatition_list[0], self.expansion, self.is_Bottleneck, stride=1 )
        self.block2 = self._make_blocks( self.channels_list[0]*self.expansion , self.channels_list[1], self.repeatition_list[1], self.expansion, self.is_Bottleneck, stride=2 )
        self.block3 = self._make_blocks( self.channels_list[1]*self.expansion , self.channels_list[2], self.repeatition_list[2], self.expansion, self.is_Bottleneck, stride=2 )
        self.block4 = self._make_blocks( self.channels_list[2]*self.expansion , self.channels_list[3], self.repeatition_list[3], self.expansion, self.is_Bottleneck, stride=2 )

        self.average_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear( self.channels_list[3]*self.expansion , num_classes)



    def forward(self,x):
        x = self.relu(self.batchnorm1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.block1(x)

        x = self.block2(x)

        x = self.block3(x)

        x = self.block4(x)

        x = self.average_pool(x)

        x = torch.flatten(x, start_dim=1)
        x = self.fc1(x)

        return x

    def _make_blocks(self,in_channels,intermediate_channels,num_repeat, expansion, is_Bottleneck, stride):

        """
        Args:
            in_channels : #channels of the Bottleneck input
            intermediate_channels : #channels of the 3x3 in the Bottleneck
            num_repeat : #Bottlenecks in the block
            expansion : factor by which intermediate_channels are multiplied to create the output channels
            is_Bottleneck : status if Bottleneck in required
            stride : stride to be used in the first Bottleneck conv 3x3

        Attributes:
            Sequence of Bottleneck layers

        """
        layers = []

        layers.append(Bottleneck(in_channels,intermediate_channels,expansion,is_Bottleneck,stride=stride))
        for num in range(1,num_repeat):
            layers.append(Bottleneck(intermediate_channels*expansion,intermediate_channels,expansion,is_Bottleneck,stride=1))

        return nn.Sequential(*layers)

class EarlyStopping:
    def __init__(self, patience=2, restore_best_weights=True):
        self.patience = patience
        self.restore_best_weights = restore_best_weights
        self.counter = 0
        self.best_loss = float('inf')
        self.best_model_weights = None
        self.early_stop = False

    def __call__(self, current_loss, model):
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.counter = 0
            if self.restore_best_weights:
                self.best_model_weights = model.state_dict().copy()
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [6]:
num_classes = len(train_dataset.classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet(model_parameters['resnet18'], in_channels=3, num_classes=num_classes)
model = model.to(device)

# define loss function & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)
early_stopping = EarlyStopping(patience=2, restore_best_weights=True)

num_epochs = 20

In [7]:
# initialize early stopping (from previous step)
early_stopping = EarlyStopping(patience=2, restore_best_weights=True)

for epoch in range(num_epochs):
    # training
    model.train()
    train_loss, train_acc = 0.0, 0.0
    start_time = time.time()

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # compute batch accuracy
        _, preds = torch.max(outputs, 1)
        acc = (preds == labels).float().mean()

        # accumulate metrics
        train_loss += loss.item() * images.size(0)
        train_acc += acc.item() * images.size(0)

        # update progress bar
        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "accuracy": f"{acc.item():.4f}"
        })

    # calculate epoch-level metrics
    train_loss /= len(train_loader.dataset)
    train_acc /= len(train_loader.dataset)

    # validation
    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_acc += (preds == labels).float().sum().item()

    val_loss /= len(test_loader.dataset)
    val_acc /= len(test_loader.dataset)

    epoch_time = time.time() - start_time
    print(
        f"Epoch {epoch+1}/{num_epochs}\n"
        f"{len(train_loader)}/{len(train_loader)} "
        f"[{'=' * 20}]\n"
        f" - loss: {train_loss:.4f}\n"
        f" - accuracy: {train_acc:.4f}\n"
        f" - val_loss: {val_loss:.4f}\n"
        f" - val_accuracy: {val_acc:.4f}\n"
        f" - time: {epoch_time:.1f}s"
    )

    # early stopping check
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

# restore best weights
if early_stopping.restore_best_weights:
    model.load_state_dict(early_stopping.best_model_weights)

Epoch 1/20: 100%|██████████| 963/963 [1:08:57<00:00,  4.30s/batch, loss=3.0191, accuracy=0.2800]


Epoch 1/20
963/963 [====================]
 - loss: 3.9185
 - accuracy: 0.0749
 - val_loss: 3.8823
 - val_accuracy: 0.1019
 - time: 5167.1s


Epoch 2/20: 100%|██████████| 963/963 [03:12<00:00,  5.01batch/s, loss=2.9192, accuracy=0.3200]


Epoch 2/20
963/963 [====================]
 - loss: 3.2919
 - accuracy: 0.1752
 - val_loss: 3.1752
 - val_accuracy: 0.2053
 - time: 234.1s


Epoch 3/20: 100%|██████████| 963/963 [03:09<00:00,  5.08batch/s, loss=2.4846, accuracy=0.3600]


...

Epoch 4/20
963/963 [====================]
 - loss: 2.5128
 - accuracy: 0.3455
 - val_loss: 2.4472
 - val_accuracy: 0.3636
 - time: 230.9s


Epoch 5/20: 100%|██████████| 963/963 [03:09<00:00,  5.09batch/s, loss=1.3831, accuracy=0.6400]


Epoch 5/20
963/963 [====================]
 - loss: 2.2084
 - accuracy: 0.4125
 - val_loss: 2.7250
 - val_accuracy: 0.3083
 - time: 230.4s


Epoch 6/20: 100%|██████████| 963/963 [03:10<00:00,  5.05batch/s, loss=1.2935, accuracy=0.6800]


Epoch 6/20
963/963 [====================]
 - loss: 1.8833
 - accuracy: 0.4880
 - val_loss: 2.4877
 - val_accuracy: 0.3758
 - time: 231.9s
Early stopping triggered.


In [8]:
model_path = "/content/drive/My Drive/resnet_18_model_lr003.pth"

# saving the model
torch.save(model.state_dict(), model_path)

In [ ]:
# # test on 1 random image

# import random
# import matplotlib.pyplot as plt

# num_classes = len(train_dataset.classes)

# model = models.resnet18(num_classes=num_classes)  # Ensure num_classes matches training

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# model.load_state_dict(torch.load(model_path, map_location=device))
# model.eval()

# random_idx = random.randint(0, len(test_dataset) - 1)
# image, label = test_dataset[random_idx]

# input_image = image.unsqueeze(0).to(device)

# with torch.no_grad():
#     output = model(input_image)
#     predicted_class = torch.argmax(output, dim=1).item()

# # reverse class-to-index mapping
# idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}
# predicted_label = idx_to_class[predicted_class]

# plt.imshow(image.permute(1, 2, 0))  # convert from (C, H, W) to (H, W, C)
# plt.title(f"Predicted: {predicted_label}")
# plt.axis("off")
# plt.show()

In [ ]:
correct = 0
total = 0
all_predictions = []
all_labels = []

# perform inference on the full test dataset
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)  # get the predicted class index

        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

# compute accuracy
accuracy = (correct / total) * 100

# convert class indices to class names
predicted_labels = [idx_to_class[idx] for idx in all_predictions]
actual_labels = [idx_to_class[idx] for idx in all_labels]

# print results
print(f"Total Test Accuracy: {accuracy:.2f}%")
print("\nPredictions vs Actual Labels:")
for pred, actual in zip(predicted_labels, actual_labels):
    print(f"Predicted: {pred}, Actual: {actual}")

Total Test Accuracy: 48.20%

Predictions vs Actual Labels:
Predicted: Apple, Actual: Apple
Predicted: salad, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: pear, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: Apricot, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: orange, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: Mee rebus, Actual: Apple
Predicted: pear, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: pear, Actual: Apple
Predicted: pear, Actual: Apple
Predicted: pineapple tarts, Actual: Apple
Predicted: Soft boiled eggs, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: otak, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: pear, Actual: Apple
Predicted: milk, Actual: Apple
Predicted: Apple, Actual: Apple
Predicted: Apricot, Actual: Apple
Predicted: pear, Actual: Apple
Predicted: Apple, Actua